<a href="https://colab.research.google.com/github/deniwidi/data-science-2026/blob/main/Pertemuan7_Deni_Widi_Alfian_240401010340.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pertemuan ke-7

- Nama    : AldyaRisnandar
- NIM     : 240401010362
- Kelas   : IF405

## Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Library berhasil diimport')


: 

## Langkah 1 — Generate & Eksplorasi Dataset
Buat dataset sintetis yang mencerminkan pola gaji realistis, kemudian lakukan EDA singkat.

In [ ]:
# Generate dataset sintetis
np.random.seed(42); n = 300
pengalaman = np.random.uniform(0, 20, n)
edu        = np.random.choice([0, 1, 2], n)  # SMA=0, D3=1, S1=2
kota       = np.random.choice(['Jakarta','Surabaya','Bandung'], n)
gaji       = (3.0 + 2.2*pengalaman + 1.5*edu
              + np.where(kota=='Jakarta', 4.0, 0)
              + np.random.normal(0, 2, n))

df = pd.DataFrame({'pengalaman': pengalaman, 'edu': edu,
                   'kota': kota, 'gaji': gaji})

# EDA singkat
print('Shape:', df.shape)
print(df.describe().round(2))

# Scatter: pengalaman vs gaji
sns.scatterplot(data=df, x='pengalaman', y='gaji',
               hue='kota', palette='Set2', alpha=0.6)
plt.title('Pengalaman vs Gaji per Kota')
plt.show()



## Langkah 2 — Preprocessing
Terapkan One-Hot Encoding pada 'kota', lakukan train-test split, kemudian StandardScaler
pada fitur numerik.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# One-Hot Encoding kolom 'kota'
df = pd.get_dummies(df, columns=['kota'],
                   drop_first=True, dtype=int)
print('Kolom setelah encoding:', df.columns.tolist())

# Pisahkan fitur dan target
X = df.drop('gaji', axis=1)
y = df['gaji']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
       X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]} baris, Test: {X_test.shape[0]} baris')

# StandardScaler — fit pada training set saja
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

## Langkah 3 — Latih Model & Tampilkan Koefisien
Buat dan latih model Regresi Linear, kemudian tampilkan koefisien β dan interpretasikan maknanya.

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train_s, y_train)

# Tampilkan koefisien
print(f'β₀ (intercept): {model.intercept_:.3f}')
print()
coef_df = pd.DataFrame({
      'Fitur' : X.columns,
      'Koefisien' : model.coef_.round(3)
}).sort_values('Koefisien', ascending=False)
print(coef_df.to_string(index=False))

# Interpretasi: fitur mana yang paling berpengaruh?
# Koefisien positif → gaji naik
# Koefisien negatif → gaji turun


In [ ]:
# Visualisasi koefisien model
fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = ['#E74C3C' if v > 0 else '#3498DB' for v in coef_df['Koefisien']]
bars = ax.barh(coef_df['Fitur'], coef_df['Koefisien'],
               color=bar_colors, edgecolor='white', height=0.5)
ax.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, coef_df['Koefisien']):
    offset = 0.15 if val >= 0 else -0.15
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:+.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=10, fontweight='bold')
ax.set_xlabel('Koefisien (skala terstandarisasi)')
ax.set_title('Koefisien Model Regresi Linear\n(merah=pengaruh positif, biru=negatif)',
             fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## Langkah 4 — Evaluasi Model
Hitung ketiga metrik evaluasi, interpretasikan setiap nilainya, dan bandingkan RMSE dengan
MAE untuk mendeteksi outlier.
from sklearn.metrics import (mean_a


In [ ]:
from sklearn.metrics import (mean_absolute_error,
     mean_squared_error, r2_score)
import numpy as np # Ensure numpy is imported

y_pred = model.predict(X_test_s)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print('=== Metrik Evaluasi ===')
print(f'MAE = {mae:.3f} juta rupiah')
print(f'RMSE = {rmse:.3f} juta rupiah')
print(f'R² = {r2:.4f} ({r2*100:.1f}% variasi dijelaskan)')
print(f'Selisih RMSE-MAE = {rmse-mae:.3f}')

# Interpretasikan dalam Markdown sel berikutnya:
# - Berapa rata-rata kesalahan prediksi dalam rupiah?
# - Apakah R² cukup baik untuk kasus ini?
# - Apakah ada indikasi outlier?

## Langkah 5 — Visualisasi & Interpretasi
Buat dua plot standar evaluasi regresi: Actual vs Predicted dan Residual Plot. Interpretasikan
setiap plot menggunakan pendekatan What? So what? Now what?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Actual vs Predicted
axes[0].scatter(y_test, y_pred, alpha=0.5,
               color='#028090', edgecolors='white', lw=0.5)
lims = [min(y_test.min(), y_pred.min()),
       max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', lw=2,
            label='Prediksi Sempurna')
axes[0].set_xlabel('Gaji Aktual (juta Rp)')
axes[0].set_ylabel('Gaji Prediksi (juta Rp)')
axes[0].set_title(f'Actual vs Predicted R²={r2:.3f}')
axes[0].legend()

# Plot 2: Residual Plot
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5,
                color='#880E4F', edgecolors='white', lw=0.5)
axes[1].axhline(0, color='gray', linestyle='--', lw=2)
axes[1].set_xlabel('Nilai Prediksi ŷ (juta Rp)')
axes[1].set_ylabel('Residual = y − ŷ (juta Rp)')
axes[1].set_title('Residual Plot')

plt.suptitle('Evaluasi Model Regresi Linear — Prediksi Gaji',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('evaluasi_regresi.png', dpi=150, bbox_inches='tight')
plt.show()

# Cara membaca Residual Plot:
# ✅ Baik: titik tersebar ACAK di sekitar garis y=0
# ❌ Masalah: ada pola (kurva, corong) → asumsi linearitas dilanggar